# Day 063 — Solution: Payments & Productization

In [ ]:
_GATED_API_SRC = '"""gated_api.py — Day 063: feature gating + Stripe payments.\n\nSetup (once):\n  pip install stripe\n  export STRIPE_SECRET_KEY=sk_test_...\n  export STRIPE_WEBHOOK_SECRET=whsec_...\n\nRun:\n  uvicorn gated_api:app --reload\nDocs:\n  http://localhost:8000/docs\n"""\nimport os\nimport stripe\nfrom datetime import datetime\nfrom fastapi import FastAPI, HTTPException, Request\nfrom pydantic import BaseModel, Field\n\nstripe.api_key = os.environ.get("STRIPE_SECRET_KEY", "")\nWEBHOOK_SECRET = os.environ.get("STRIPE_WEBHOOK_SECRET", "")\nAPP_VER        = "1.0.0"\n\nPLAN_PRICES = {\n    "pro": os.environ.get("STRIPE_PRICE_PRO", "price_pro_monthly"),\n}\n\nFEATURE_MATRIX = {\n    "free": {"basic_chat", "view_history"},\n    "pro":  {"basic_chat", "view_history", "advanced_chat", "export", "api_access"},\n}\n\nDAILY_LIMITS = {"free": 10, "pro": 1_000}\n\n\ndef check_feature_access(plan: str, feature: str) -> bool:\n    return feature in FEATURE_MATRIX.get(plan, set())\n\n\ndef check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:\n    limit = DAILY_LIMITS.get(plan, 0)\n    if usage_count >= limit:\n        return False, f"Daily limit reached for {plan!r} plan ({usage_count}/{limit})"\n    return True, ""\n\n\nimport ollama\n\nMODEL = os.environ.get("MODEL", "llama3.2")\n\n\ndef build_api(process_fn=None, initial_plan: str = "free",\n              initial_usage: int = 0) -> FastAPI:\n    """Build the gated API.\n\n    process_fn: optional callable(prompt: str) -> str for testing.\n    """\n    app   = FastAPI(title="Gated API", version=APP_VER)\n    _state = {"plan": initial_plan, "usage": initial_usage}\n\n    class AskRequest(BaseModel):\n        prompt: str = Field(min_length=1)\n\n    class CheckoutRequest(BaseModel):\n        plan: str\n\n    @app.get("/health")\n    def health():\n        return {"status": "ok", "timestamp": datetime.utcnow().isoformat() + "Z"}\n\n    @app.get("/plan")\n    def get_plan():\n        plan  = _state["plan"]\n        limit = DAILY_LIMITS.get(plan, 0)\n        return {"plan": plan, "usage_today": _state["usage"], "limit": limit}\n\n    @app.post("/ask")\n    def ask(req: AskRequest):\n        plan    = _state["plan"]\n        allowed, reason = check_rate_limit(_state["usage"], plan)\n        if not allowed:\n            raise HTTPException(status_code=429, detail=reason)\n        if process_fn is not None:\n            answer = process_fn(req.prompt)\n        else:\n            resp   = ollama.chat(\n                model=MODEL,\n                messages=[{"role": "user", "content": req.prompt}],\n            )\n            answer = resp["message"]["content"]\n        _state["usage"] += 1\n        return {"answer": answer, "plan": plan, "requests_remaining":\n                DAILY_LIMITS.get(plan, 0) - _state["usage"]}\n\n    @app.post("/checkout")\n    def create_checkout(req: CheckoutRequest):\n        if req.plan not in PLAN_PRICES:\n            raise HTTPException(status_code=400,\n                                detail=f"Unknown plan: {req.plan!r}")\n        session = stripe.checkout.Session.create(\n            payment_method_types=["card"],\n            line_items=[{"price": PLAN_PRICES[req.plan], "quantity": 1}],\n            mode="subscription",\n            success_url="http://localhost:8000/checkout/success?session_id={CHECKOUT_SESSION_ID}",\n            cancel_url="http://localhost:8000/checkout/cancel",\n        )\n        return {"session_id": session["id"], "checkout_url": session["url"]}\n\n    @app.get("/checkout/success")\n    def checkout_success(session_id: str):\n        _state["plan"] = "pro"\n        _state["usage"] = 0\n        return {"success": True, "plan": "pro", "session_id": session_id}\n\n    @app.get("/checkout/cancel")\n    def checkout_cancel():\n        return {"message": "Checkout cancelled. No charges were made."}\n\n    @app.post("/webhook")\n    async def stripe_webhook(request: Request):\n        payload   = await request.body()\n        sig       = request.headers.get("stripe-signature", "")\n        try:\n            event = stripe.Webhook.construct_event(payload, sig, WEBHOOK_SECRET)\n        except (stripe.error.SignatureVerificationError, ValueError):\n            raise HTTPException(status_code=400, detail="Invalid signature")\n        etype = event["type"]\n        if etype in ("customer.subscription.created", "customer.subscription.updated"):\n            _state["plan"] = "pro"\n        elif etype == "customer.subscription.deleted":\n            _state["plan"] = "free"\n        return {"received": True, "type": etype}\n\n    return app\n\n\napp = build_api()\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
from pathlib import Path
Path('gated_api.py').write_text(_GATED_API_SRC)
print('gated_api.py written.')

In [ ]:
# inline test — no Stripe credentials or Ollama needed
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

FEATURE_MATRIX = {
    "free": {"basic_chat", "view_history"},
    "pro":  {"basic_chat", "view_history", "advanced_chat", "export", "api_access"},
}
DAILY_LIMITS = {"free": 10, "pro": 1_000, "enterprise": float("inf")}

def check_feature_access(plan, feature):
    return feature in FEATURE_MATRIX.get(plan, set())

def check_rate_limit(usage_count, plan):
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit:
        return False, f"Daily limit reached for {plan!r} plan"
    return True, ""

# --- inline gated API ---
def build_api(process_fn=None, initial_plan="free", initial_usage=0):
    app = FastAPI(); _s = {"plan": initial_plan, "usage": initial_usage}
    class _R(BaseModel):
        prompt: str = Field(min_length=1)
    @app.get("/plan")
    def plan():
        lim = DAILY_LIMITS.get(_s["plan"], 0)
        return {"plan": _s["plan"], "usage_today": _s["usage"],
                "limit": lim if lim != float("inf") else -1}
    @app.post("/ask")
    def ask(req: _R):
        ok, reason = check_rate_limit(_s["usage"], _s["plan"])
        if not ok: raise HTTPException(429, reason)
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        _s["usage"] += 1
        return {"answer": answer, "plan": _s["plan"]}
    return app

# tests
c = TestClient(build_api(process_fn=str.upper), raise_server_exceptions=False)

assert c.get("/plan").json()["plan"] == "free"
print("\u2705 /plan returns free")

r = c.post("/ask", json={"prompt": "hello"})
assert r.status_code == 200 and r.json()["answer"] == "HELLO"
print("\u2705 /ask works")

assert c.post("/ask", json={"prompt": ""}).status_code == 422
print("\u2705 empty prompt \u2192 422")

c2 = TestClient(build_api(process_fn=str.upper, initial_usage=10),
                raise_server_exceptions=False)
assert c2.post("/ask", json={"prompt": "x"}).status_code == 429
print("\u2705 free at limit \u2192 429")

# feature access checks
assert check_feature_access("free", "basic_chat") is True
assert check_feature_access("free", "advanced_chat") is False
assert check_feature_access("pro", "advanced_chat") is True
print("\u2705 check_feature_access correct")

# rate limit checks
assert check_rate_limit(5, "free") == (True, "")
assert check_rate_limit(10, "free")[0] is False
assert check_rate_limit(999_999, "enterprise")[0] is True
print("\u2705 check_rate_limit correct")

# webhook event processing
PLAN_EVENTS = {"customer.subscription.created": "pro",
               "customer.subscription.deleted": "free"}
def process_webhook_event(etype, payload, user_db):
    cid = payload.get("customer_id", "")
    if not cid: return {"success": False, "action": "no_customer_id", "customer_id": ""}
    if cid not in user_db: return {"success": False, "action": "customer_not_found", "customer_id": cid}
    if etype in PLAN_EVENTS:
        np = PLAN_EVENTS[etype]; user_db[cid]["plan"] = np
        return {"success": True, "action": f"plan_set_{np}", "customer_id": cid}
    if etype == "invoice.payment_failed":
        user_db[cid]["status"] = "past_due"
        return {"success": True, "action": "status_set_past_due", "customer_id": cid}
    return {"success": True, "action": "ignored", "customer_id": cid}

db = {"cus_1": {"plan": "free", "status": "active"}}
assert process_webhook_event("customer.subscription.created", {"customer_id": "cus_1"}, db)["action"] == "plan_set_pro"
assert db["cus_1"]["plan"] == "pro"
print("\u2705 webhook event processing correct")

print("\nDay 063 \u2014 Payments & Productization complete! \U0001f389")
